# Random Forest

For this project we will be exploring publicly available data from [LendingClub.com](www.lendingclub.com). Lending Club connects people who need money (borrowers) with people who have money (investors). Hopefully, as an investor you would want to invest in people who showed a profile of having a high probability of paying you back. We will try to create a model that will help predict this.

Lending club had a [very interesting year in 2016](https://en.wikipedia.org/wiki/Lending_Club#2016), so let's check out some of their data and keep the context in mind. This data is from before they even went public.

We will use lending data from 2007-2010 and be trying to classify and predict whether or not the borrower paid back their loan in full. You can download the data from [here](https://www.lendingclub.com/info/download-data.action) or just use the csv already provided. It's recommended you use the csv provided as it has been cleaned of NA values.

Here are what the columns represent:
* credit.policy: 1 if the customer meets the credit underwriting criteria of LendingClub.com, and 0 otherwise.
* purpose: The purpose of the loan (takes values "credit_card", "debt_consolidation", "educational", "major_purchase", "small_business", and "all_other").
* int.rate: The interest rate of the loan, as a proportion (a rate of 11% would be stored as 0.11). Borrowers judged by LendingClub.com to be more risky are assigned higher interest rates.
* installment: The monthly installments owed by the borrower if the loan is funded.
* log.annual.inc: The natural log of the self-reported annual income of the borrower.
* dti: The debt-to-income ratio of the borrower (amount of debt divided by annual income).
* fico: The FICO credit score of the borrower.
* days.with.cr.line: The number of days the borrower has had a credit line.
* revol.bal: The borrower's revolving balance (amount unpaid at the end of the credit card billing cycle).
* revol.util: The borrower's revolving line utilization rate (the amount of the credit line used relative to total credit available).
* inq.last.6mths: The borrower's number of inquiries by creditors in the last 6 months.
* delinq.2yrs: The number of times the borrower had been 30+ days past due on a payment in the past 2 years.
* pub.rec: The borrower's number of derogatory public records (bankruptcy filings, tax liens, or judgments).
* not.fully.paid: 1 if the loan is not paid in full, and 0 otherwise.

## Learning Objectives

At the end of this notebook, you should be able to:

- Explore the Lending Club data with histograms, a countplot, and faceted regression plots, and interpret what each one shows.
- Encode the categorical `purpose` feature with `pd.get_dummies` and split the data into training and test sets.
- Train and compare a decision tree and a random forest to predict loan defaults.
- Evaluate both models with a classification report and confusion matrix, counter class imbalance with `class_weight`, and rank feature importances.

## Import Libraries

**Import the usual libraries for pandas and plotting. You can import sklearn later on.**

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams["patch.force_edgecolor"] = True

## Get the Data

**Use pandas to read loan_data.csv as a dataframe called loans.**

In [ ]:
loans = pd.read_csv('../data/loan_data.csv')

**Check out the info(), head(), and describe() methods on loans.**

In [ ]:
print(loans.info())

In [ ]:
loans.describe()

In [ ]:
loans.head()

## Exploratory Data Analysis

Let's do some data visualization! We'll use seaborn and pandas built-in plotting capabilities, but feel free to use whatever library you want. Don't worry about the colors matching, just worry about getting the main idea of the plot.

**Create a histogram of two FICO distributions on top of each other, one for each credit.policy outcome.**

*Note: This is pretty tricky, feel free to reference the solutions. You'll probably need one line of code for each histogram, I also recommend just using pandas built in .hist()*

In [ ]:
plt.figure(figsize=(10,5))
loans[loans['credit.policy']==1]['fico'].hist(bins=30, color='blue', label='Policy: 0', alpha=0.6)
loans[loans['credit.policy']==0]['fico'].hist(bins=30, color='red', label='Policy: 1', alpha=0.6)
plt.legend()

**Create a similar figure, except this time select by the not.fully.paid column.**

In [ ]:
plt.figure(figsize=(10,6))
loans[loans['not.fully.paid']==0]['fico'].hist(bins=30, color='blue', label='Fully Paid', alpha=0.4)
loans[loans['not.fully.paid']==1]['fico'].hist(bins=30, color='red', label='Not fully Paid', alpha=0.5)
plt.legend()

**Create a countplot using seaborn showing the counts of loans by purpose, with the color hue defined by not.fully.paid.**

In [ ]:
plt.figure(figsize = (11,6))
sns.countplot(x=loans['purpose'], hue=loans['not.fully.paid'], palette = 'Set1')

<details>
<summary>Show answer</summary>

`debt_consolidation` and `credit_card` loans dominate the raw counts, so they also contribute the largest absolute number of not-fully-paid loans. But volume is not the same as risk: `small_business` loans carry a much higher *share* of not-fully-paid loans relative to their total, so they look disproportionately risky. That is a more useful read than simply noting that there are a lot of debt consolidation loans.
</details>

**Let's see the trend between FICO score and interest rate. Recreate the following jointplot.**

In [ ]:
sns.jointplot(x='fico', y='int.rate', data=loans, kind='hex')

**The `jointplot` above showed the overall FICO vs interest-rate trend. Now let's ask a sharper question: does that trend differ depending on *why* people borrow?**

**Use `sns.lmplot()` to plot `int.rate` against `fico`, faceting into one column per loan `purpose` (use `col='purpose'` with `col_wrap=3`). Pass `scatter_kws={'alpha': 0.3}` so overlapping points stay readable.**

This is more informative than splitting on `credit.policy`, which is itself derived from FICO, so a FICO split there mostly just re-draws the same line. Loan purpose is independent of the score, so the per-purpose regression lines can genuinely differ.

> Reminder: `lmplot()` is a scatter plot with an overlaid regression line (`regplot()`), wrapped in a `FacetGrid()` so you can break the relationship out across subsets. See the [lmplot() docs](https://seaborn.pydata.org/generated/seaborn.lmplot.html) and the [regplot vs lmplot explainer](https://seaborn.pydata.org/tutorial/regression.html).

In [ ]:
sns.lmplot(x='fico', y='int.rate', data=loans, col='purpose', col_wrap=3,
           height=3, scatter_kws={'alpha': 0.3})

<details>
<summary>Show answer</summary>

Across every purpose the relationship is negative: higher FICO scores get lower interest rates, which is what we would expect. The interesting part is that the slope is not identical everywhere. `small_business` and `debt_consolidation` are priced more steeply against credit score, while thinner categories such as `educational` are flatter and noisier (partly because they have far fewer loans, so the regression band is wide). This is more honest than the single `jointplot`: the overall hex plot hid the fact that pricing depends on *why* someone is borrowing, not just on their score. Splitting on `credit.policy` instead would not reveal this, because `credit.policy` is itself a FICO-based rule, so both groups would trace essentially the same line.
</details>

## Setting up the Data

Let's get ready to set up our data for our Random Forest Classification Model!

**Check loans.info() again.**

In [ ]:
loans.info()

### Categorical Features

Notice that the **purpose** column as categorical

That means we need to transform them using dummy variables so sklearn will be able to understand them. Let's do this in one clean step using pd.get_dummies.

Let's show you a way of dealing with these columns that can be expanded to multiple categorical features if necessary.

**Create a list of 1 element containing the string 'purpose'. Call this list cat_feats.**

In [ ]:
cat_feats = ['purpose']

In [ ]:
cat_feats

**Now use pd.get_dummies(loans,columns=cat_feats,drop_first=True) to create a fixed larger dataframe that has new feature columns with dummy variables. Set this dataframe as final_data.**

In [ ]:
final_data = pd.get_dummies(loans, columns=cat_feats, drop_first=True)

In [ ]:
final_data.head()

### Train Test Split

Now its time to split our data into a training set and a testing set!

**Use sklearn to split your data into a training set and a testing set as we've done in the past.**

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X = final_data.drop('not.fully.paid', axis=1)
y = final_data['not.fully.paid']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101)

## Training a Decision Tree Model

Let's start by training a single decision tree first!

**Import DecisionTreeClassifier**

In [ ]:
from sklearn.tree import DecisionTreeClassifier

**Create an instance of DecisionTreeClassifier() called dtree and fit it to the training data.**

In [ ]:
dtree = DecisionTreeClassifier()

In [ ]:
dtree.fit(X_train, y_train)

### Predictions and Evaluation of Decision Tree
**Create predictions from the test set and create a classification report and a confusion matrix.**

In [ ]:
predictions = dtree.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
print(classification_report(y_test, predictions))

In [ ]:
print(confusion_matrix(y_test, predictions))

## Training the Random Forest model

Now its time to train our model!

**Create an instance of the RandomForestClassifier class and fit it to our training data from the previous step.**

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
rnc = RandomForestClassifier(n_estimators = 200)

In [ ]:
rnc.fit(X_train, y_train)

### Predictions and Evaluation

Let's predict off the y_test values and evaluate our model.

**Predict the class of not.fully.paid for the X_test data.**

In [ ]:
pred = rnc.predict(X_test)

**Now create a classification report from the results. Do you get anything strange or some sort of warning?**

In [ ]:
print(classification_report(y_test, pred))

**Show the Confusion Matrix for the predictions.**

In [ ]:
print(confusion_matrix(y_test, pred))

## Interpreting your Results

**The random forest has the higher overall accuracy, but what happened to its recall for class 1 (loans that were not fully paid)? Which model is actually more useful to an investor, and why is accuracy misleading here?**

<details>
<summary>Show answer</summary>

The single decision tree caught about a quarter of the risky loans (recall around 0.24 for class 1), at the cost of more false alarms. The random forest scored higher overall accuracy (about 0.84 vs 0.73) but its recall for class 1 collapsed to roughly 0.02: it predicts "fully paid" for almost everyone. Because only 16% of loans default, always predicting "fully paid" already scores about 84% accuracy, so accuracy rewards exactly the wrong behaviour. An investor cares about catching the bad loans, which is recall on class 1, so the high-accuracy forest is of little practical use as trained. That is the problem the next step fixes.
</details>

### Dealing With Class Imbalance

Only about 16% of the loans in this dataset were not fully paid, so a model can score high accuracy by almost always predicting "fully paid". One simple fix is to tell the model to weight the rare class more heavily with `class_weight='balanced'`.

**Retrain the random forest with `class_weight='balanced'` (keep `n_estimators=200` and set `random_state=101` for reproducibility), predict on `X_test`, and compare the classification report and confusion matrix to the unbalanced version.**

In [ ]:
rnc_balanced = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=101)
rnc_balanced.fit(X_train, y_train)
pred_balanced = rnc_balanced.predict(X_test)

print(classification_report(y_test, pred_balanced))
print(confusion_matrix(y_test, pred_balanced))

<details>
<summary>Show answer</summary>

Balancing the class weights nudges the model to pay more attention to the rare class. Recall for class 1 rises from about 0.02 to roughly 0.12: still low, but we now flag around 55 of the true defaulters instead of 9. The cost is the expected trade-off: precision on class 1 stays weak (about 0.29) and overall accuracy slips a little (from about 0.84 to 0.82) because we raise more false alarms on loans that were actually repaid. `class_weight='balanced'` is a modest adjustment, not a complete fix: pushing recall meaningfully higher would need resampling, threshold tuning, or better features. The real lesson is that a single accuracy number hid all of this.
</details>

### Which Features Drive the Prediction?

A random forest exposes how much each feature contributed to its splits through `.feature_importances_`. Plotting them turns an opaque model into something we can check against our EDA.

**Build a sorted pandas Series of the forest's feature importances and plot it as a horizontal bar chart.**

In [ ]:
importances = pd.Series(rnc.feature_importances_, index=X.columns).sort_values()

plt.figure(figsize=(10, 6))
importances.plot(kind='barh')
plt.title('Random Forest Feature Importances')
plt.xlabel('Importance')
plt.tight_layout()

<details>
<summary>Show answer</summary>

Two things stand out. First, the continuous numeric columns (`installment`, `revol.util`, `days.with.cr.line`, `revol.bal`, `dti`, `log.annual.inc`) cluster at the top, while the one-hot `purpose_*` dummies and rare flags like `pub.rec` and `credit.policy` sit near the bottom. Second, and more surprising: `fico` and `int.rate`, the variables that looked most discriminative in our EDA, land only in the middle of the pack (`int.rate` around 7th, `fico` around 8th).

That mismatch is the real lesson. Default (Gini) feature importance in a random forest is biased toward features with many distinct values, so high-cardinality continuous columns get inflated and a coarser but genuinely predictive feature like `fico` can be understated. Treat this chart as a hint, not the final word: permutation importance or SHAP values would give a fairer ranking. "The model says X is important" always deserves a second look.
</details>

## The End!